# Qwen . Kaggle

Сравним исходный промпт и промпт с более строгим правилом для кадров без бабочки.

Нужно создать Kaggle Dataset с одним файлом `butterfly_validation.json` и подключить его через **Add Input**.

In [ ]:
from pathlib import Path

DATA_PATH = None  # Можно указать полный путь к файлу в Kaggle.
WORK = Path("/kaggle/working")

if DATA_PATH is None:
    matches = sorted(Path("/kaggle/input").rglob("butterfly_validation.json"))
    if len(matches) != 1:
        raise ValueError(
            "Через Add Input подключи датасет с butterfly_validation.json. "
            f"Найдено файлов: {len(matches)}. Если их несколько, укажи DATA_PATH вручную."
        )
    DATA_PATH = matches[0]
else:
    DATA_PATH = Path(DATA_PATH)
print("Датасет:", DATA_PATH)


Датасет: /kaggle/input/datasets/hmossya/butterflies-dataset1/butterfly_validation.json


## Проверяем датасет

Собираем фото и эталонные рамки по `photo_id`. Эталон используем при оценке, а моделям передаём изображения и промпт.

In [2]:
import hashlib
import json
import math

def check_boxes(boxes):
    if not isinstance(boxes, list):
        raise ValueError("boxes должен быть списком")
    for box in boxes:
        x, y, w, h = (box[key] for key in ("left", "top", "width", "height"))
        if not all(type(v) in (float, int) and math.isfinite(v) for v in (x, y, w, h)):
            raise ValueError(f"Неверные координаты: {box}")
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
                and x+w <= 1+1e-9 and y+h <= 1+1e-9):
            raise ValueError(f"Рамка выходит за границы или имеет нулевой размер: {box}")


rows = json.loads(DATA_PATH.read_text(encoding="utf-8-sig"))
team = {}
golden = {}
for row in rows:
    photo_id = row["photo_id"]
    if type(photo_id) is not int or photo_id <= 0 or photo_id in team:
        raise ValueError(f"Неверный или повторный photo_id: {photo_id!r}")
    url, group = row["url"].strip(), row["group"].strip()
    if not url.startswith(("https://", "http://")) or not group:
        raise ValueError(f"Проверь url и group у {photo_id}")
    check_boxes(row["boxes"])
    team[photo_id] = {"image": url, "group": group}
    golden[photo_id] = {"photo_id": photo_id, "boxes": row["boxes"]}
if len(team) != 400:
    raise ValueError(f"Ожидается 400 уникальных photo_id, получено {len(team)}")
dataset_sha256 = hashlib.sha256(
    json.dumps([team, golden], sort_keys=True).encode()
).hexdigest()
print(f"Проверено фото: {len(team)} | с эталонными рамками: {sum(bool(r['boxes']) for r in golden.values())}")


Проверено фото: 400 | с эталонными рамками: 197


## Считаем метрики

Одна предсказанная рамка может совпасть только с одной эталонной. Проверяем два порога перекрытия, 0.50 и 0.75.

In [3]:
from decimal import Decimal

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

def iou_matrix(reference, predicted):
    def corners(boxes):
        xywh = np.asarray([
            [box["left"], box["top"], box["width"], box["height"]] for box in boxes
        ], dtype=float).reshape(-1, 4)
        return xywh[:, :2], xywh[:, :2] + xywh[:, 2:], np.prod(xywh[:, 2:], axis=1)

    ref_min, ref_max, ref_area = corners(reference)
    pred_min, pred_max, pred_area = corners(predicted)
    top_left = np.maximum(ref_min[:, None], pred_min[None, :])
    bottom_right = np.minimum(ref_max[:, None], pred_max[None, :])
    intersection = np.maximum(bottom_right - top_left, 0).prod(axis=2)
    union = ref_area[:, None] + pred_area[None, :] - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)


def summarize(rows, suffix):
    tp, fp, fn = (sum(row[f"{key}_{suffix}"] for row in rows) for key in ("tp", "fp", "fn"))
    return {
        f"tp_{suffix}": tp,
        f"fp_{suffix}": fp,
        f"fn_{suffix}": fn,
        f"precision_{suffix}": tp / (tp + fp) if tp + fp else 0.0,
        f"recall_{suffix}": tp / (tp + fn) if tp + fn else 0.0,
        f"f1_{suffix}": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
    }


def evaluate(photo_ids, golden, predictions):
    rows = []
    for photo_id in photo_ids:
        reference = golden[photo_id]["boxes"]
        record = predictions[photo_id]
        error = record.get("error")
        predicted = [] if error else record["boxes"]
        ious = iou_matrix(reference, predicted)
        row = {"photo_id": photo_id, "is_negative": not reference, "error": error}
        for threshold, suffix in ((0.5, "50"), (0.75, "75")):
            tp = 0
            if ious.size:
                matches = maximum_bipartite_matching(csr_matrix(ious >= threshold))
                tp = int(np.count_nonzero(matches >= 0))
            row.update({
                f"tp_{suffix}": tp, f"fp_{suffix}": len(predicted) - tp,
                f"fn_{suffix}": len(reference) - tp,
            })
        rows.append(row)

    summary = {"n_photos": len(rows), "technical_errors": sum(bool(row["error"]) for row in rows)}
    for suffix in ("50", "75"):
        summary.update(summarize(rows, suffix))
    negatives = [row for row in rows if row["is_negative"]]
    summary.update(
        n_negative_photos=len(negatives),
        negative_photos_with_fp=sum(row["fp_50"] > 0 for row in negatives),
        fp_on_negative_photos=sum(row["fp_50"] for row in negatives),
    )
    return summary, rows


def quality_score(f1):
    value = Decimal(str(f1))
    if not 0 <= value <= 1:
        raise ValueError("F1 must be in [0, 1]")
    return int(min(Decimal(10), 10 * (value + Decimal("0.3"))))


box = {"left": 0.1, "top": 0.2, "width": 0.3, "height": 0.4}
test_rows = {1: {"boxes": [box], "error": None}}
assert evaluate(test_rows, test_rows, test_rows)[0]["f1_50"] == 1.0


## Настройки эксперимента

In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Config:
    name: str
    backend: str
    model_id: str
    prompt: str | tuple[str, ...]
    revision: str = "main"
    candidate_threshold: float = 0.0
    threshold: float = 0.0
    text_threshold: float = 0.25
    max_new_tokens: int = 512
    image_max_side: int = 1024
    load_in_4bit: bool = False


MODEL = "qwen"
REVISION = "main"
NEGATIVE_RULE = (
    "Include a box only if the visible anatomy clearly identifies an adult butterfly or moth. "
    "On a flower-only image or an image containing only other insects, return []."
)

config = Config(
    'qwen',
    'qwen',
    'Qwen/Qwen3-VL-2B-Instruct',
    (
        'Detect every visible adult butterfly or moth. Give each individual one tight '
        'bounding box covering only its visible parts. Ignore other insects, '
        'caterpillars, pupae, and shadows. Return only a JSON array of objects with '
        '"bbox_2d": [x_min, y_min, x_max, y_max], using integer coordinates from 0 to '
        '1000 relative to the full image. Return [] if no adult butterfly or moth is '
        'visible.'
    ),
    revision=REVISION,
)
reference_path = WORK / f"{MODEL}__reference.jsonl"
variants = {}
settings = {}


## Готовим GPU и зависимости

In [5]:
import os
import subprocess
import sys

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.1", "huggingface-hub>=0.34,<1", "accelerate>=1,<2",
], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Включи GPU в настройках среды и запусти ячейки заново.")
device = "cuda"
print(torch.cuda.get_device_name(0))


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


Tesla T4


## Загружаем модель и обрабатываем фото

Сырые ответы сохраняются после каждого фото. При повторе в той же среде используем уже успешные ответы.

In [6]:
import gc
import re
from dataclasses import asdict, replace
from io import BytesIO
from time import perf_counter
from urllib.request import urlopen

from PIL import Image, ImageOps

def normalized_box(coords, width, height):
    """Convert XYXY coordinates to the contest format; ignore empty boxes."""
    x1, y1, x2, y2 = coords
    if not all(math.isfinite(value) for value in coords):
        raise ValueError("Non-finite box coordinates")
    left, right = (min(1.0, max(0.0, x / width)) for x in (x1, x2))
    top, bottom = (min(1.0, max(0.0, y / height)) for y in (y1, y2))
    if right <= left or bottom <= top:
        return None
    return dict(left=left, top=top, width=right-left, height=bottom-top)


def parse_qwen(text):
    """Parse the whole response, accepting an optional Markdown JSON fence."""
    fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text.strip(), re.S | re.I)
    items = json.loads(fenced.group(1) if fenced else text)
    if not isinstance(items, list):
        raise ValueError("Expected a JSON list")
    boxes = []
    for item in items:
        coords = item.get("bbox_2d") if isinstance(item, dict) else None
        if not isinstance(coords, list) or len(coords) != 4 or not all(
            type(value) in (int, float) and 0 <= value <= 1000 for value in coords
        ):
            raise ValueError("Expected four bbox_2d coordinates in [0, 1000]")
        box = normalized_box(coords, 1000, 1000)
        if box is None:
            raise ValueError("Qwen returned an empty or reversed box")
        boxes.append(box)
    return boxes


class QwenResponseError(ValueError):
    def __init__(self, message, raw_response):
        super().__init__(message)
        self.raw_response = raw_response


def load_detector(config, device):
    import torch
    from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor
    from transformers import BitsAndBytesConfig

    processor = AutoProcessor.from_pretrained(config.model_id, revision=config.revision)
    if config.backend == "qwen":
        from transformers import Qwen3VLForConditionalGeneration
        dtype = torch.float16 if device.startswith("cuda") else torch.float32
        options = {"dtype": dtype}
        if config.load_in_4bit:
            if not device.startswith("cuda"):
                raise ValueError("This 4-bit configuration requires a CUDA device")
            options["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype,
            )
            options["device_map"] = {"": device}
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            config.model_id, revision=config.revision, **options
        )
        if not config.load_in_4bit:
            model.to(device)
    else:
        model = AutoModelForZeroShotObjectDetection.from_pretrained(
            config.model_id, revision=config.revision
        ).to(device)
    return processor, model.eval()


def predict_one(detector, config, image, device):
    """Return candidates (each with a box and optional score) and model text."""
    import torch
    from PIL import Image

    processor, model = detector
    with torch.inference_mode():
        if config.backend == "qwen":
            resized = image.copy()
            resized.thumbnail(
                (config.image_max_side, config.image_max_side), Image.Resampling.LANCZOS
            )
            messages = [{"role": "user", "content": [
                {"type": "image", "image": resized},
                {"type": "text", "text": config.prompt},
            ]}]
            chat_text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            inputs = processor(
                text=[chat_text], images=[resized], return_tensors="pt"
            ).to(device)
            ids = model.generate(
                **inputs, max_new_tokens=config.max_new_tokens, do_sample=False
            )
            text = processor.batch_decode(
                ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
            )[0]
            try:
                boxes = parse_qwen(text)
            except ValueError as error:
                raise QwenResponseError(str(error), text) from error
            return [{"box": box, "score": None} for box in boxes], text

        size = [(image.height, image.width)]
        query = config.prompt if config.backend == "dino" else [list(config.prompt)]
        inputs = processor(images=image, text=query, return_tensors="pt").to(device)
        outputs = model(**inputs)
        if config.backend == "dino":
            result = processor.post_process_grounded_object_detection(
                outputs, input_ids=inputs.input_ids, target_sizes=size,
                threshold=config.candidate_threshold, text_threshold=config.text_threshold,
            )[0]
        else:
            result = processor.image_processor.post_process_object_detection(
                outputs, target_sizes=size, threshold=config.candidate_threshold
            )[0]

    candidates = []
    for coords, score in zip(result["boxes"].tolist(), result["scores"].tolist()):
        box = normalized_box(coords, image.width, image.height)
        if box is not None:
            candidates.append({"box": box, "score": score})
    return candidates, None


def select_boxes(candidates, threshold):
    return [
        candidate["box"] for candidate in candidates
        if candidate["score"] is None or candidate["score"] >= threshold
    ]


def open_image(source):
    if source.startswith(("https://", "http://")):
        with urlopen(source, timeout=30) as response:
            source = BytesIO(response.read())
    with Image.open(source) as image:
        return ImageOps.exif_transpose(image).convert("RGB")


def read_jsonl(path):
    rows = {}
    with Path(path).open(encoding="utf-8-sig") as file:
        for line in file:
            if not line.strip():
                continue
            row = json.loads(line)
            record_id = row.get("photo_id")
            if type(record_id) is not int or record_id in rows:
                raise ValueError(f"{path}: неверный или повторный photo_id {record_id!r}")
            rows[record_id] = row
    return rows


def read_settings(raw_path):
    path = Path(raw_path).with_suffix(".settings.json")
    return json.loads(path.read_text(encoding="utf-8"))


def run_model(config, manifest, raw_path, device):
    """Сохраняет кандидатов после каждого фото. При повторе обрабатывает сбои."""
    import torch
    import transformers
    from huggingface_hub import model_info

    raw_path = Path(raw_path)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    settings_path = raw_path.with_suffix(".settings.json")
    previous = read_settings(raw_path) if settings_path.exists() else None
    if raw_path.exists() and previous is None:
        raise ValueError(f"Для {raw_path} отсутствует файл настроек")

    if config.revision == "main":
        revision = previous["config"]["revision"] if previous else model_info(config.model_id).sha
        config = replace(config, revision=revision)

    settings = {
        "config": asdict(config),
        "device": device,
        "gpu": torch.cuda.get_device_name(device) if device.startswith("cuda") else None,
        "torch": str(torch.__version__),
        "transformers": transformers.__version__,
        "manifest_hash": hashlib.sha256(
            json.dumps(manifest, sort_keys=True).encode("utf-8")
        ).hexdigest(),
    }
    settings_text = json.dumps(settings, ensure_ascii=False, indent=2)
    if previous is not None and previous != json.loads(settings_text):
        raise ValueError("Настройки или набор фото изменились. Укажи новый путь для результатов.")
    if previous is None:
        settings_path.write_text(settings_text, encoding="utf-8")

    results = read_jsonl(raw_path) if raw_path.exists() else {}
    pending = [pid for pid in manifest if pid not in results or results[pid]["error"]]
    if not pending:
        return results

    print(f"{config.name}: загружаю {config.model_id} ({config.revision[:10]}) на {device}", flush=True)
    detector = load_detector(config, device)
    print(f"{config.name}: модель загружена, осталось фото: {len(pending)}", flush=True)
    try:
        for index, photo_id in enumerate(pending, 1):
            previous_attempts = results.get(photo_id, {}).get("attempt", 0)
            record = {"photo_id": photo_id, "attempt": previous_attempts + 1}
            try:
                image = open_image(manifest[photo_id]["image"])
                if device.startswith("cuda"):
                    torch.cuda.synchronize(device)
                started = perf_counter()
                candidates, text = predict_one(detector, config, image, device)
                if device.startswith("cuda"):
                    torch.cuda.synchronize(device)
                record.update(candidates=candidates, raw_response=text,
                              seconds=perf_counter() - started, error=None)
            except Exception as error:
                record.update(candidates=None, seconds=None,
                              raw_response=getattr(error, "raw_response", None),
                              error=f"{type(error).__name__}: {error}")

            if record["error"] or previous_attempts:
                log_path = raw_path.with_suffix(".errors.jsonl")
                with log_path.open("a", encoding="utf-8") as log:
                    log.write(json.dumps(record, ensure_ascii=False) + "\n")

            results[photo_id] = record
            write_jsonl(raw_path, results.values())
            if index == 1 or index % 25 == 0 or index == len(pending):
                print(f"{config.name}: {index}/{len(pending)}", flush=True)
    finally:
        del detector
        gc.collect()
        if device.startswith("cuda"):
            torch.cuda.empty_cache()
    return results


def export_predictions(raw, manifest, threshold, path):
    if set(raw) != set(manifest):
        raise ValueError("ID результатов и манифеста должны совпадать")
    rows = {}
    for photo_id in manifest:
        record = raw[photo_id]
        boxes = None if record["error"] else select_boxes(record["candidates"], threshold)
        rows[photo_id] = {"photo_id": photo_id, "boxes": boxes, "error": record["error"]}
    write_jsonl(path, rows.values())
    return rows


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
    temp.replace(path)


### Исходная настройка

In [14]:
def find_kaggle_file(filename, required=True):
    matches = sorted(Path("/kaggle/input").rglob(filename))

    if len(matches) > 1:
        paths = "\n".join(str(path) for path in matches)
        raise ValueError(f"Найдено несколько файлов {filename}:\n{paths}")

    if matches:
        return matches[0]

    if required:
        raise FileNotFoundError(
            f"Файл {filename} не найден. Подключи его датасет через Add Input."
        )

    return None


baseline_path = find_kaggle_file("qwen.jsonl")
settings_path = find_kaggle_file("qwen.settings.json", required=False)

print(f"Файл с предсказаниями: {baseline_path}")
reference = read_jsonl(baseline_path)

if reference.keys() != team.keys():
    raise ValueError("photo_id в qwen.jsonl и датасете должны совпадать")

if any("boxes" not in row for row in reference.values()):
    raise ValueError("Подключи qwen.jsonl из results/predictions/ с полем boxes")

if settings_path is not None:
    print(f"Файл настроек: {settings_path}")
    saved_settings = json.loads(settings_path.read_text(encoding="utf-8-sig"))
    config = replace(config, **saved_settings["config"])
    settings["reference"] = saved_settings
else:
    settings["reference"] = {"config": asdict(config)}
    print("qwen.settings.json не найден. Ревизия весов бейзлайна не подтверждена.")

write_jsonl(reference_path, reference.values())
print(f"Загружены готовые предсказания Qwen для {len(reference)} фото.")

Датасет: /kaggle/input/datasets/hmossya/butterflies-dataset1/butterfly_validation.json


In [20]:
def find_kaggle_file(filename, required=True):
    matches = sorted(Path("/kaggle/input").rglob(filename))

    if len(matches) > 1:
        paths = "\n".join(str(path) for path in matches)
        raise ValueError(f"Найдено несколько файлов {filename}:\n{paths}")

    if matches:
        return matches[0]

    if required:
        raise FileNotFoundError(
            f"Файл {filename} не найден. Подключи его датасет через Add Input."
        )

    return None


baseline_path = find_kaggle_file("qwen.jsonl")
settings_path = find_kaggle_file("qwen.settings.json", required=False)

print(f"Файл с предсказаниями: {baseline_path}")
reference = read_jsonl(baseline_path)

if reference.keys() != team.keys():
    raise ValueError("photo_id в qwen.jsonl и датасете должны совпадать")

if any("boxes" not in row for row in reference.values()):
    raise ValueError("Подключи qwen.jsonl из results/predictions/ с полем boxes")

if settings_path is not None:
    print(f"Файл настроек: {settings_path}")
    saved_settings = json.loads(settings_path.read_text(encoding="utf-8-sig"))
    config = replace(config, **saved_settings["config"])
    settings["reference"] = saved_settings
else:
    settings["reference"] = {"config": asdict(config)}
    print("qwen.settings.json не найден. Ревизия весов бейзлайна не подтверждена.")

write_jsonl(reference_path, reference.values())
print(f"Загружены готовые предсказания Qwen для {len(reference)} фото.")

Файл с предсказаниями: /kaggle/input/datasets/hmossya/qwen-predictions/qwen.jsonl
Файл настроек: /kaggle/input/datasets/hmossya/qwen-sett/qwen.settings.json
Загружены готовые предсказания Qwen для 400 фото.


### Экспериментальная настройка

In [21]:
label = "qwen__strict_negative_prompt"
changed_config = replace(
    config,
    name=label,
    revision=settings["reference"]["config"]["revision"],
    prompt=config.prompt + " " + NEGATIVE_RULE,
)
raw_path = WORK / f"{label}_raw.jsonl"
changed_raw = run_model(changed_config, team, raw_path, device)
settings[label] = read_settings(raw_path)
export_predictions(changed_raw, team, changed_config.threshold, WORK / f"{label}.jsonl")
variants[label] = {"factor": "prompt", "value": NEGATIVE_RULE}


qwen__strict_negative_prompt: загружаю Qwen/Qwen3-VL-2B-Instruct (89644892e4) на cuda
qwen__strict_negative_prompt: модель загружена, осталось фото: 400
qwen__strict_negative_prompt: 1/400
qwen__strict_negative_prompt: 25/400
qwen__strict_negative_prompt: 50/400
qwen__strict_negative_prompt: 75/400
qwen__strict_negative_prompt: 100/400
qwen__strict_negative_prompt: 125/400
qwen__strict_negative_prompt: 150/400
qwen__strict_negative_prompt: 175/400
qwen__strict_negative_prompt: 200/400
qwen__strict_negative_prompt: 225/400
qwen__strict_negative_prompt: 250/400
qwen__strict_negative_prompt: 275/400
qwen__strict_negative_prompt: 300/400
qwen__strict_negative_prompt: 325/400
qwen__strict_negative_prompt: 350/400
qwen__strict_negative_prompt: 375/400
qwen__strict_negative_prompt: 400/400


## Сравниваем результаты

In [22]:
base, _ = evaluate(team, golden, reference)
summary = [{"run": reference_path.stem, **base, "delta_f1_50": 0.0,
            "validation_score_10": quality_score(base["f1_50"])}]
for label, variant in variants.items():
    predictions = read_jsonl(WORK / f"{label}.jsonl")
    stats, _ = evaluate(team, golden, predictions)
    summary.append({
        "run": label, **variant, **stats,
        "delta_f1_50": stats["f1_50"] - base["f1_50"],
        "delta_negative_photos_with_fp": (
            stats["negative_photos_with_fp"] - base["negative_photos_with_fp"]
        ),
        "validation_score_10": quality_score(stats["f1_50"]),
    })

board = pd.DataFrame(summary)
display(board[[
    "run", "f1_50", "f1_75", "precision_50", "recall_50", "delta_f1_50",
    "negative_photos_with_fp", "technical_errors", "validation_score_10",
]])
if board["technical_errors"].any():
    print("Есть технические ошибки. Перед отправкой повтори ячейки с прогоном. "
          "Успешные фото возьмутся из сохранённых результатов.")


,run,f1_50,f1_75,precision_50,recall_50,delta_f1_50,negative_photos_with_fp,technical_errors,validation_score_10
0,qwen__reference,0.650794,0.584127,0.497573,0.940367,0.000000,202,0,9
1,qwen__strict_negative_prompt,0.874200,0.788913,0.816733,0.940367,0.223407,43,0,10


## Скачиваем результаты

Этот ZIP отдаём для сборки общего лидерборда.

In [23]:
from zipfile import ZIP_DEFLATED, ZipFile

metadata_path = WORK / f"{MODEL}.experiment.json"
metadata = {
    "model": MODEL,
    "dataset_sha256": dataset_sha256,
    "reference": reference_path.name,
    "variants": variants,
    "settings": settings,
}
metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
csv_path = WORK / f"{MODEL}_experiment.csv"
board.to_csv(csv_path, index=False)

archive = WORK / f"{MODEL}_experiment_results.zip"
variant_paths = [WORK / f"{label}.jsonl" for label in variants]
with ZipFile(archive, "w", ZIP_DEFLATED) as bundle:
    for path in [reference_path, *variant_paths, metadata_path, csv_path]:
        bundle.write(path, arcname=path.name)

from IPython.display import FileLink
display(FileLink(archive.name))


/kaggle/working/qwen_experiment_results.zip